<a href="https://colab.research.google.com/github/aashish003/AI-lab/blob/main/1BF24CS003_Week7_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
class Rule:
    def __init__(self, premises, conclusion):
        # Premises is a list of tuples: (predicate_name, tuple_of_arguments)
        self.premises = premises
        # Conclusion is a single tuple: (predicate_name, tuple_of_arguments)
        self.conclusion = conclusion

    def __str__(self):
        premises_str = " & ".join([f"{p[0]}({', '.join(p[1])})" for p in self.premises])
        return f"{premises_str} => {self.conclusion[0]}({', '.join(self.conclusion[1])})"


def unify(pattern, fact, theta):
    """
    Unifies a rule pattern (with variables) and a ground fact.
    Returns updated substitution dictionary if unification is successful, else None.
    """
    if len(pattern) != len(fact):
        return None

    new_theta = theta.copy()
    for p_arg, f_arg in zip(pattern, fact):
        # We consider any string starting with a lowercase letter to be a variable
        is_var = p_arg[0].islower()

        if is_var:
            if p_arg in new_theta:
                if new_theta[p_arg] != f_arg:
                    return None
            else:
                new_theta[p_arg] = f_arg
        else:
            if p_arg != f_arg:
                return None
    return new_theta


def match_premises(premises, current_kb, theta):
    """
    Recursively matches premises against existing ground facts in KB.
    Yields valid substitutions (theta) that satisfy all premises.
    """
    if not premises:
        yield theta
        return

    first, rest = premises[0], premises[1:]
    pred_name, pattern_args = first

    for fact_pred, fact_args in current_kb:
        if pred_name == fact_pred:
            new_theta = unify(pattern_args, fact_args, theta)
            if new_theta is not None:
                yield from match_premises(rest, current_kb, new_theta)


def forward_chaining_solver(initial_facts, rules, goal_pred, goal_args):
    """
    Generalized Forward Chaining Algorithm.
    Takes any defined facts, rules, and goal to find a proof.
    """
    # Represent KB as a set of ground facts (predicate, tuple_of_constants)
    kb = set(initial_facts)
    goal = (goal_pred, goal_args)

    print("--- Generalized Forward Chaining ---")
    print("\nInitial Facts:")
    for fact in sorted(kb):
        print(f"  {fact[0]}({', '.join(fact[1])})")

    print("\nRules:")
    for i, rule in enumerate(rules, start=1):
        print(f"  Rule {i}: {rule}")

    print(f"\nGoal: {goal[0]}({', '.join(goal[1])})\n")

    step = 1
    new_facts_inferred = True

    while new_facts_inferred:
        new_facts_inferred = False
        new_inferences = set()

        for rule in rules:
            # Search for valid substitutions matching all premises of the rule
            for substitution in match_premises(rule.premises, kb, {}):
                conc_pred, conc_args = rule.conclusion
                # Apply substitution mapping variables (lowercase) to constants
                substituted_args = tuple(substitution.get(arg, arg) for arg in conc_args)
                inferred_fact = (conc_pred, substituted_args)

                if inferred_fact not in kb and inferred_fact not in new_inferences:
                    new_inferences.add(inferred_fact)
                    print(f"[Step {step}] Inferred: {conc_pred}({', '.join(substituted_args)})")
                    print(f"         via Rule [{rule}] with substitution {substitution}\n")
                    step += 1
                    new_facts_inferred = True

        if new_inferences:
            kb.update(new_inferences)

        # Check if the goal state is reached
        if goal in kb:
            print("SUCCESS: Goal has been proved!")
            return True

    print("FAILURE: No more inferences possible. Goal could not be proved.")
    return False


# =========================================================
# USER INPUT DEFINITIONS (Define any facts, rules, and goal)
# =========================================================

# 1. Define your initial facts: (predicate, (constants, ...))
custom_facts = [
    ("American", ("Robert",)),
    ("Enemy", ("A", "America")),
    ("Owns", ("A", "T1")),
    ("Missile", ("T1",))
]

# 2. Define your Rules using Rule(premises, conclusion)
# Variables must start with a lowercase letter (e.g., 'x', 'p', 'q')
# Constants must start with an uppercase letter or number (e.g., 'Robert', 'T1', 'A')
custom_rules = [
    Rule([
        ("Enemy", ("x", "America"))
    ], ("Hostile", ("x",))),

    Rule([
        ("Missile", ("x",))
    ], ("Weapon", ("x",))),

    Rule([
        ("Missile", ("x",)),
        ("Owns", ("A", "x"))
    ], ("Sells", ("Robert", "x", "A"))),

    Rule([
        ("American", ("p",)),
        ("Weapon", ("q",)),
        ("Sells", ("p", "q", "r")),
        ("Hostile", ("r",))
    ], ("Criminal", ("p",)))
]

# 3. Define the query goal to prove: (predicate, (arguments, ...))
goal_predicate = "Criminal"
goal_arguments = ("Robert",)

# Run the solver
forward_chaining_solver(custom_facts, custom_rules, goal_predicate, goal_arguments)

--- Generalized Forward Chaining ---

Initial Facts:
  American(Robert)
  Enemy(A, America)
  Missile(T1)
  Owns(A, T1)

Rules:
  Rule 1: Enemy(x, America) => Hostile(x)
  Rule 2: Missile(x) => Weapon(x)
  Rule 3: Missile(x) & Owns(A, x) => Sells(Robert, x, A)
  Rule 4: American(p) & Weapon(q) & Sells(p, q, r) & Hostile(r) => Criminal(p)

Goal: Criminal(Robert)

[Step 1] Inferred: Hostile(A)
         via Rule [Enemy(x, America) => Hostile(x)] with substitution {'x': 'A'}

[Step 2] Inferred: Weapon(T1)
         via Rule [Missile(x) => Weapon(x)] with substitution {'x': 'T1'}

[Step 3] Inferred: Sells(Robert, T1, A)
         via Rule [Missile(x) & Owns(A, x) => Sells(Robert, x, A)] with substitution {'x': 'T1'}

[Step 4] Inferred: Criminal(Robert)
         via Rule [American(p) & Weapon(q) & Sells(p, q, r) & Hostile(r) => Criminal(p)] with substitution {'p': 'Robert', 'q': 'T1', 'r': 'A'}

SUCCESS: Goal has been proved!


True

In [ ]:
custom_facts = [
    ("Dog", ("Rex",)),
    ("LivesInHouse", ("Rex",))
]

# 2. Define your Rules using Rule(premises, conclusion)
# Variables must start with a lowercase letter (e.g., 'x', 'p', 'q')
# Constants must start with an uppercase letter or number (e.g., 'Robert', 'T1', 'A')
custom_rules = [
    Rule([
        ("Dog", ("x",))
    ], ("Mammal", ("x",))),

    Rule([
        ("Mammal", ("x",)),
        ("LivesInHouse", ("x",))
    ], ("Pet", ("x",)))
]

# 3. Define the query goal to prove: (predicate, (arguments, ...))
goal_predicate = "Pet"
goal_arguments = ("Rex",)

# Run the solver
forward_chaining_solver(custom_facts, custom_rules, goal_predicate, goal_arguments)